In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/mixalldata_clean.csv')

df.head()

,type,sendTime,sender,senderPseudo,messageID,class,posx,posy,posz,posx_n,...,aclz,aclx_n,acly_n,aclz_n,hedx,hedy,hedz,hedx_n,hedy_n,hedz_n
0,4,72002.302942,130137,101301377,422013806,0,266.982401,32.336955,0.0,3.480882,...,0.0,0.000862,0.000862,0.0,-0.102790,0.994703,0.0,20.038218,17.541001,0.0
1,4,72003.302942,130137,101301377,422023410,0,266.827208,34.624145,0.0,3.546261,...,0.0,0.000107,0.001040,0.0,-0.099856,0.995002,0.0,20.441139,14.467283,0.0
2,4,72004.302942,130137,101301377,422032081,0,266.420297,38.836461,0.0,3.544045,...,0.0,0.000172,0.001661,0.0,-0.099856,0.995002,0.0,20.850473,11.941528,0.0
3,4,72005.302942,130137,101301377,422040712,0,268.912026,45.414229,0.0,3.340080,...,0.0,0.000171,0.001654,0.0,-0.100172,0.994970,0.0,21.323229,9.633965,0.0
4,4,72006.302942,130137,101301377,422052949,0,268.242276,53.729986,0.0,3.328872,...,0.0,0.000193,0.001852,0.0,-0.097105,0.995274,0.0,21.788034,7.824555,0.0


In [2]:
cols_to_drop = ['type', 'sender', 'senderPseudo', 'messageID', 
                'sendTime', 'posz', 'posz_n', 'spdz', 'spdz_n',
                'aclz', 'aclz_n', 'hedz', 'hedz_n']
df = df.drop(columns=cols_to_drop)

print(df.columns.tolist())

['class', 'posx', 'posy', 'posx_n', 'posy_n', 'spdx', 'spdy', 'spdx_n', 'spdy_n', 'aclx', 'acly', 'aclx_n', 'acly_n', 'hedx', 'hedy', 'hedx_n', 'hedy_n']


In [3]:
df['attack'] = (df['class'] > 0).astype(int)

print('Distribution binaire :')
print(df['attack'].value_counts())

Distribution binaire :
attack
0    1900539
1    1294269
Name: count, dtype: int64


In [4]:
print(f'Colonnes après nettoyage : {df.shape[1]} colonnes')
df.columns

Colonnes après nettoyage : 18 colonnes


Index(['class', 'posx', 'posy', 'posx_n', 'posy_n', 'spdx', 'spdy', 'spdx_n',
       'spdy_n', 'aclx', 'acly', 'aclx_n', 'acly_n', 'hedx', 'hedy', 'hedx_n',
       'hedy_n', 'attack'],
      dtype='object')

In [5]:
from sklearn.preprocessing import StandardScaler

cols_to_exclude = ['class', 'attack']
features_to_scale = df.select_dtypes(
    include='number').columns.difference(cols_to_exclude).tolist()

scaler = StandardScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

In [6]:
print(f'Features normalisées : {features_to_scale}')
df[features_to_scale].describe().loc[['mean', 'std']]

Features normalisées : ['aclx', 'aclx_n', 'acly', 'acly_n', 'hedx', 'hedx_n', 'hedy', 'hedy_n', 'posx', 'posx_n', 'posy', 'posy_n', 'spdx', 'spdx_n', 'spdy', 'spdy_n']


,aclx,aclx_n,acly,acly_n,hedx,hedx_n,hedy,hedy_n,posx,posx_n,posy,posy_n,spdx,spdx_n,spdy,spdy_n
mean,1.400265e-17,-1.569471e-16,-7.658754e-17,-3.471304e-17,-3.745308e-18,9.818934e-16,9.038558e-18,-5.522061e-16,1.892315e-15,-2.441549e-15,2.188114e-16,2.169930e-15,1.731026e-16,-6.120598e-18,2.917960e-18,8.042181e-18
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['class', 'attack'])
y_binary = df['attack']
y_multi = df['class']

X_train, X_test, y_train_binary, y_test_binary = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

_, _, y_train_multi, y_test_multi = train_test_split(
    X, y_multi, test_size=0.2, random_state=42
)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')

X_train : (2555846, 16)
X_test  : (638962, 16)


In [8]:
print(f'Taille du dataset complet  : {X.shape[0]:,} lignes')
print(f'Taille du train            : {X_train.shape[0]:,} lignes (80%)')
print(f'Taille du test             : {X_test.shape[0]:,} lignes (20%)')

Taille du dataset complet  : 3,194,808 lignes
Taille du train            : 2,555,846 lignes (80%)
Taille du test             : 638,962 lignes (20%)


In [9]:
import os, json

os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train_binary.to_csv('../data/processed/y_train_binary.csv', index=False)
y_test_binary.to_csv('../data/processed/y_test_binary.csv', index=False)
y_train_multi.to_csv('../data/processed/y_train_multi.csv', index=False)
y_test_multi.to_csv('../data/processed/y_test_multi.csv', index=False)

class_mapping = {
    0: 'Normal', 1: 'Constant position', 2: 'Constant position offset',
    3: 'Random position', 4: 'Random position offset', 5: 'Constant speed',
    6: 'Constant speed offset', 7: 'Random speed', 8: 'Random speed offset',
    9: 'Disruptive', 10: 'Data replay', 11: 'DoS', 12: 'DoS random',
    13: 'DoS disruptive', 14: 'Data replay sybil', 15: 'Traffic congestion sybil',
    16: 'DoS random sybil', 17: 'DoS disruptive sybil', 18: 'Random speed',
    19: 'Random speed offset'
}

with open('../data/processed/class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=4)

print('Sauvegarde terminée:')
print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')

Sauvegarde terminée:
X_train : (2555846, 16)
X_test  : (638962, 16)


In [11]:
print(X_train.columns.tolist())

['posx', 'posy', 'posx_n', 'posy_n', 'spdx', 'spdy', 'spdx_n', 'spdy_n', 'aclx', 'acly', 'aclx_n', 'acly_n', 'hedx', 'hedy', 'hedx_n', 'hedy_n']
